# Region → zone: learned + structural + local search

같은 데이터와 checkpoint 예측으로 `learned`, `learned_structural`, 그리고 각각의 count-NLL local search 결과를 비교합니다. 128과 256은 **별도로 학습**합니다. 기존 `learned` 모델/학습 loss는 유지합니다.

기본 실험: train 1600 / validation 100 / test 200 tables, 각 table당 sequence 1개, 50 epochs, seed 42. GPU는 새 모델 학습에 사용하고 작은 permutation의 local search는 CPU에서 수행합니다. 먼저 짧게 확인하려면 아래 `LENGTHS = [128]`로 설정하세요.

**기존 `raw_results.json`이 있다면** 아래 `SOURCE_RESULTS`에 경로를 넣으세요. 저장된 예측을 이용하므로 재학습과 GPU 없이도 보정 전후를 비교할 수 있습니다.

결과와 checkpoint는 Google Drive에 저장하고 마지막에 공유할 ZIP을 만듭니다. NLL 개선이 정답률 개선을 보장하지 않으므로 worse table 수도 함께 확인하세요.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
repo = Path('/content/matrix')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/lyh4215/matrix.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

LENGTHS = [128, 256]
TRAIN_TABLES = 1600
VALIDATION_TABLES = 100
TEST_TABLES = 200
EPOCHS = 50
SEED = 42
SEARCH_RESTARTS = 1  # 먼저 learned 초기값 하나의 효과를 측정
INCLUDE_ORACLE = False  # True면 CPU oracle baseline도 같은 test tables로 재계산
SOURCE_RESULTS = []  # 예: ['/content/drive/MyDrive/old_run/raw_results.json']
OUTPUT_ROOT = Path('/content/drive/MyDrive/matrix_region_zone')


In [ ]:
import datetime
import json
import torch

if not SOURCE_RESULTS:
    assert torch.cuda.is_available(), 'Colab 런타임의 하드웨어 가속기를 GPU로 바꾸세요.'
RUN_DIR = OUTPUT_ROOT / datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
RUN_DIR.mkdir(parents=True, exist_ok=False)
metadata = {
    'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'torch': str(torch.__version__),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'lengths': LENGTHS, 'seed': SEED, 'epochs': EPOCHS,
    'source_results': SOURCE_RESULTS,
}
(RUN_DIR / 'environment.json').write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

def run_logged(command, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    print('Running:', ' '.join(command), flush=True)
    with (output_dir / 'console.log').open('w') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)

if SOURCE_RESULTS:
    for index, source in enumerate(SOURCE_RESULTS):
        output = RUN_DIR / f'replay_{index}'
        run_logged([sys.executable, 'region_zone_refine.py', '--results', str(source),
                    '--restarts', str(SEARCH_RESTARTS), '--output-dir', str(output)], output)
else:
    matchers = ['learned', 'learned_structural']
    if INCLUDE_ORACLE:
        matchers.insert(0, 'oracle_transition')
    for length in LENGTHS:
        output = RUN_DIR / f'length_{length}'
        run_logged([
            sys.executable, 'region_zone_match_probe.py', '--matchers', *matchers,
            '--train-tables', str(TRAIN_TABLES), '--validation-tables', str(VALIDATION_TABLES),
            '--test-tables', str(TEST_TABLES), '--sequence-lengths', str(length),
            '--epochs', str(EPOCHS), '--batch-size', '8', '--seed', str(SEED), '--device', 'cuda',
            '--structural-refinement-steps', '4', '--structural-beta', '0.1', '--lambda-graph', '0.5',
            '--local-search', '--local-search-restarts', str(SEARCH_RESTARTS),
            '--output-dir', str(output),
        ], output)


In [ ]:
import shutil
from IPython.display import Markdown, display

for summary in sorted(RUN_DIR.glob('*/summary.md')):
    display(Markdown(summary.read_text()))
archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('공유할 결과 ZIP:', archive)
print('정답률 delta, worse tables, NLL gain, search time을 함께 비교하세요.')


In [ ]:
# 선택: 결과 ZIP을 다운로드해서 대화에 첨부할 수 있습니다.
from google.colab import files
files.download(archive)
